In [4]:
import open3d as o3d
import numpy as np
import os
os.chdir("C:\\Users\\lambo\\OneDrive - HKUST Connect\\1_Projects\ASTRI\\Project folder\\Hole detection")
print(os.getcwd())

# Load mesh
mesh = o3d.io.read_triangle_mesh("./ply_files/temp/reconstructed_mesh_stomach1.ply")
mesh = o3d.io.read_triangle_mesh("./ply_files/temp/ball_pivoting_mesh_.ply")
# mesh = o3d.io.read_triangle_mesh("./ply_files/temp/reconstructed_mesh_hourglass_closed.ply")
# mesh = o3d.io.read_triangle_mesh("./ply_files/temp/reconstructed_mesh_snail6000.ply")
# Compute AABB
aabb = mesh.get_axis_aligned_bounding_box()
aabb.color = (1, 0, 0)  # Red for AABB

# Get dimensions (width, height, depth)
aabb_extent = aabb.get_extent()  # Returns (width, height, depth)
print(f"AABB Dimensions (W x H x D): {aabb_extent}")

# Visualize
o3d.visualization.draw_geometries([mesh, aabb], window_name="AABB")

C:\Users\lambo\OneDrive - HKUST Connect\1_Projects\ASTRI\Project folder\Hole detection
AABB Dimensions (W x H x D): [2.21251941 1.76252341 2.29680061]


### Find holes


In [6]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA
from scipy.spatial import Delaunay

def compute_alpha_shape_area(points, alpha):
    tri = Delaunay(points)
    # Filter triangles based on circumradius
    valid_tris = [t for t in tri.simplices 
                 if np.linalg.norm(np.mean(points[t], axis=0) - points[t[0]]) < alpha]
    return sum(0.5 * np.linalg.norm(np.cross(
        points[t[1]] - points[t[0]], 
        points[t[2]] - points[t[0]])) for t in valid_tris)

# Load your mesh (reconstructed from previous steps)
mesh = mesh
# mesh = o3d.io.read_triangle_mesh("./ply_files/reconstructed/hole_example.ply")

# Add mesh cleaning before hole detection
mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles()
mesh.remove_degenerate_triangles()
mesh.remove_non_manifold_edges()

# 1. Hole Detection Function
def detect_and_measure_holes(mesh, min_hole_area=0.01):
    # Create wireframe for hole detection
    wireframe = o3d.geometry.LineSet.create_from_triangle_mesh(mesh)
    
    # Find boundary edges (potential holes)
    boundaries = o3d.geometry.LineSet()
    boundary_edges = []
    
    # Get all edges and their triangle counts
    triangles = np.asarray(mesh.triangles)
    edges = {}
    for tri in triangles:
        edges[tuple(sorted((tri[0], tri[1])))] = edges.get(tuple(sorted((tri[0], tri[1]))), 0) + 1
        edges[tuple(sorted((tri[1], tri[2])))] = edges.get(tuple(sorted((tri[1], tri[2]))), 0) + 1
        edges[tuple(sorted((tri[2], tri[0])))] = edges.get(tuple(sorted((tri[2], tri[0]))), 0) + 1
    
    # Identify boundary edges (edges with only one triangle)
    boundary_edges = [edge for edge, count in edges.items() if count == 1]
    
    # Build graph for hole identification
    # Builds a graph where vertices are connected by boundary edges.
    graph = {}
    for edge in boundary_edges:
        v1, v2 = edge
        graph.setdefault(v1, []).append(v2)
        graph.setdefault(v2, []).append(v1)
    
    # Find connected components (potential holes)
    '''
    This code snippet is designed to traverse a graph and identify "holes," which are likely 
    cycles or loops containing at least three vertices. The outer loop iterates over each vertex
     in the graph. For every vertex that hasn't been visited yet, it initializes a stack with 
     that vertex and an empty list called hole to keep track of the current cycle.

    The inner while loop performs a depth-first search (DFS) using the stack. It pops a vertex 
    from the stack, checks if it has been visited, and if not, marks it as visited and adds it 
    to the current hole. Then, it extends the stack with all unvisited neighbors of the current 
    vertex, retrieved using graph.get(current, []).
    '''
    # A subtle point is that this approach does not explicitly check for cycles; 
    # it collects all reachable vertices from each unvisited starting point. 
    # For true cycle detection, additional logic would be needed to ensure the path forms a 
    # closed loop.
    visited = set()
    holes = []
    for vertex in graph:
        if vertex not in visited:
            stack = [vertex]
            hole = []
            while stack:
                current = stack.pop()
                if current not in visited:
                    visited.add(current)
                    hole.append(current)
                    stack.extend([n for n in graph.get(current, []) if n not in visited])
            if len(hole) >= 1:  # Only consider loops with at least 3 vertices
                holes.append(hole)
    
    # 2. Hole Measurement and Filtering
    valid_holes = []
    hole_geometries = []
    hole_meshes = []
    hole_areas = []

    for hole in holes:
        # Get hole vertices
        hole_vertices = np.asarray(mesh.vertices)[hole]
        
        # Fit plane to hole points
        """
        To analyze the hole's shape, the code fits a plane to the hole's 
        vertices using Principal Component Analysis (PCA) with three 
        components. The normal vector of the fitted plane is taken as the
        smallest principal component, which represents the direction with
        the least variance among the points.
        """
        pca = PCA(n_components=3) # 3D space has three principal components
        pca.fit(hole_vertices)
        normal = pca.components_[2]  # Smallest principal component
        
        # Project hole to "statistical orthogonal" plane
        centered = hole_vertices - pca.mean_
        projected = centered - np.outer(np.dot(centered, normal), normal)
        
        # Compute convex hull area (connecting the points) from projected points
        
        # two ways to compute area (1. convex hull, alpha shape area)
        hull = ConvexHull(projected[:, :2])  # Use first two principal components
        area = hull.volume  # For 2D, volume = area 
        # area = compute_alpha_shape_area(projected, alpha=0.1)
        hole_areas.append(area)

        # Get hole centroid
        centroid = np.mean(hole_vertices, axis=0)
        
        # Filter by area
        if area >= min_hole_area:
            valid_holes.append({
                "vertices": hole_vertices,
                "area": area,
                "centroid": centroid,
                "boundary_points": hole_vertices[hull.vertices]
            })
            
        # Create a watertight mesh for the hole
        try:
            # Create triangulation using convex hull
            hole_mesh = o3d.geometry.TriangleMesh()
            hole_mesh.vertices = o3d.utility.Vector3dVector(hole_vertices[hull.vertices])
            
            # Create triangles by connecting all boundary points to centroid
            triangles = []
            num_points = len(hole_vertices[hull.vertices])
            for i in range(num_points):
                triangles.append([i, (i+1)%num_points, num_points])  # Connect to center
            
            # Add centroid as final vertex
            vertices_with_center = np.vstack([hole_vertices[hull.vertices], centroid])
            hole_mesh.vertices = o3d.utility.Vector3dVector(vertices_with_center)
             # Create triangles with consistent winding order (outward-facing)
            for i in range(num_points):
                # Create triangle: current point, next point, centroid
                # Ensure counter-clockwise winding when viewed from outside
                triangles.append([(i+1) % num_points, i, num_points])
            
            hole_mesh.triangles = o3d.utility.Vector3iVector(triangles)
            
            # --- NORMAL CORRECTION STARTS HERE ---
            # Compute normals and ensure consistent orientation
            hole_mesh.compute_vertex_normals()
            
            # Flip normals if needed (based on convex hull normal)
            hull_normal = pca.components_[2]
            mesh_center = np.asarray(mesh.get_center())
            
            # Check orientation: vector from centroid to mesh center should align with hull normal
            to_center = mesh_center - centroid
            to_center /= np.linalg.norm(to_center)
            
            # If normals are pointing inward, flip all triangles
            if np.dot(hull_normal, to_center) > 0:
                hole_mesh.triangles = o3d.utility.Vector3iVector(
                    np.asarray(hole_mesh.triangles)[:, ::-1]  # Reverse winding order
                )
                hole_mesh.compute_vertex_normals()
            # --- NORMAL CORRECTION ENDS HERE ---
            
            # Make the hole surface clearly visible
            hole_mesh.paint_uniform_color([1, 0, 0])  # Bright red
            
            # Smooth the hole surface for better visualization
            hole_mesh = hole_mesh.filter_smooth_taubin(number_of_iterations=5)
            hole_meshes.append(hole_mesh)
            
        except Exception as e:
            print(f"Could not create mesh for hole: {e}")
    
    return valid_holes, hole_meshes, hole_areas

# 3. Detect and measure holes
min_area = 0.0001  # Minimum hole area to consider (adjust based on your scene scale)
detected_holes, hole_meshes, hole_areas = detect_and_measure_holes(mesh, min_hole_area=min_area)

# 4. Info printing
print(f"Detected {len(detected_holes)} significant holes:")
for i, hole in enumerate(detected_holes):
    print(f"Hole {i+1}:")
    print(f"  - Area: {hole['area']:.6f}")
    print(f"  - Centroid: {hole['centroid']}")
    print(f"  - Boundary points: {len(hole['vertices'])}")
print(f"Hole areas:")
for area in hole_areas:
    print(f"  - {area:.6f}")

# Create visualization
mesh.paint_uniform_color([0.5, 0.5, 0.5])  # Gray mesh
visualization_geometries = [mesh] + hole_meshes

# Add coordinate frame for reference
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1)
visualization_geometries.append(coordinate_frame)

# Visualize
o3d.visualization.draw_geometries(
    visualization_geometries,
    window_name=f"Hole Detection (Min Area: {min_area})",
    mesh_show_wireframe=True
)

# Modified visualization section
mesh.paint_uniform_color([0.8, 0.8, 0.8])  # Light gray for main mesh
# Create visualization geometries
visualization_geometries = [mesh] + hole_meshes

# Visualize with settings that make holes clearly visible
o3d.visualization.draw_geometries(
    visualization_geometries,
    window_name=f"Hole Detection (Min Area: {min_area})",
    mesh_show_wireframe=False,
    mesh_show_back_face=False,  # Helps with depth perception
    point_show_normal=False,
    zoom=0.7  # Adjust zoom to see holes better
)

Detected 4 significant holes:
Hole 1:
  - Area: 0.048011
  - Centroid: [-0.53475968  0.64755441  0.67272672]
  - Boundary points: 81
Hole 2:
  - Area: 0.026577
  - Centroid: [-1.27351939  0.67623622  0.91732964]
  - Boundary points: 89
Hole 3:
  - Area: 0.075150
  - Centroid: [-0.67732134  0.65014639  1.2489525 ]
  - Boundary points: 81
Hole 4:
  - Area: 0.003649
  - Centroid: [-0.11659627  0.76679438  0.39391417]
  - Boundary points: 28
Hole areas:
  - 0.048011
  - 0.026577
  - 0.075150
  - 0.003649
  - 0.000075


### Find holes directly from point cloud

Processing Slices: 100%|██████████| 100/100 [00:52<00:00,  1.90slice/s]
